In [1]:
#Week 3 m3
import pandas as pd
from pathlib import Path

FILE_NAME = "GB_GBN SPECIFIC DATASET.xlsx"

DATA_PATH = Path("..") / "data" / "raw" / FILE_NAME
print("File found:", DATA_PATH.exists())

df = pd.read_excel(DATA_PATH)
df["utc_timestamp"] = pd.to_datetime(df["utc_timestamp"], utc=True)

#Shortening column names for simplification.
P = "GB_GBN_price_day_ahead"
LOAD = "GB_GBN_load_actual_entsoe_transparency"
SOLAR = "GB_GBN_solar_generation_actual"
SOLAR_CAP = "GB_GBN_solar_capacity"

print(df.shape)

File found: True
(50401, 16)


In [2]:
#Locating unrealistic values:

#Outputting the number of duplicate rows and timestamps
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df["utc_timestamp"].duplicated().sum())

#Columns with negative values
num = df.select_dtypes("number")
negatives = (num<0).sum()
print("Columns with negative values:")
print(negatives[negatives > 0])

#Locating Solar generation higher than solar capacity
print("Solar generation > solar capacity:", (df[SOLAR] > df[SOLAR_CAP]).sum())

Duplicate rows: 0
Duplicate timestamps: 0
Columns with negative values:
GB_GBN_price_day_ahead    81
dtype: int64
Solar generation > solar capacity: 233


In [3]:
#Checking profile values
profiles  = [c for c in df.columns if c.endswith("_profile")]
rows = []
for c in profiles:
    rows.append({
        "columns":c,
        "below_0": int((df[c] <0).sum()),
        "above_1": int((df[c] >1).sum()),
        "max": round(df[c].max(),3),
    })
pd.DataFrame(rows)

,columns,below_0,above_1,max
0,GB_GBN_solar_profile,0,233,1.367
1,GB_GBN_wind_profile,0,0,0.755
2,GB_GBN_wind_offshore_profile,0,0,0.881
3,GB_GBN_wind_onshore_profile,0,0,0.760


In [4]:
#Unusual day ahead prices
print(df[P].describe().round(2))
print("Negative prices:", (df[P] <0).sum())
print("Prices >= 200:", (df[P] >= 200).sum())
print("Prices >= 500:", (df[P] >= 500).sum())

q1,q3 = df[P].quantile([0.25,0.75])
iqr = q3 - q1
lo,hi = q1 - 1.5 * iqr,q3 +1.5* iqr
print("IQR bounds:",round(lo,1), "to", round(hi,1))
print("Rows outside IQR bounds:",((df[P] < lo) | (df[P] > hi)).sum())


count    50290.00
mean        43.43
std         18.83
min        -38.80
25%         33.88
50%         41.01
75%         50.40
max        999.00
Name: GB_GBN_price_day_ahead, dtype: float64
Negative prices: 81
Prices >= 200: 29
Prices >= 500: 10
IQR bounds: 9.1 to 75.2
Rows outside IQR bounds: 1804


In [5]:
#10 most expensive hours
df.nlargest(10,P)[["utc_timestamp",P]]


,utc_timestamp,GB_GBN_price_day_ahead
14971,2016-09-15 18:00:00+00:00,999.00
15067,2016-09-19 18:00:00+00:00,960.00
14972,2016-09-15 19:00:00+00:00,799.00
16242,2016-11-07 17:00:00+00:00,796.00
16266,2016-11-08 17:00:00+00:00,775.92
15066,2016-09-19 17:00:00+00:00,750.00
15068,2016-09-19 19:00:00+00:00,747.00
16410,2016-11-14 17:00:00+00:00,700.00
16290,2016-11-09 17:00:00+00:00,600.01
15065,2016-09-19 16:00:00+00:00,500.00


In [6]:
#10 Least expensive hours
df.nsmallest(10,P)[["utc_timestamp",P]]


,utc_timestamp,GB_GBN_price_day_ahead
47271,2020-05-23 14:00:00+00:00,-38.80
47261,2020-05-23 04:00:00+00:00,-36.60
47260,2020-05-23 03:00:00+00:00,-34.93
47270,2020-05-23 13:00:00+00:00,-34.00
47269,2020-05-23 12:00:00+00:00,-28.80
47262,2020-05-23 05:00:00+00:00,-28.60
47259,2020-05-23 02:00:00+00:00,-26.05
47288,2020-05-24 07:00:00+00:00,-25.43
47285,2020-05-24 04:00:00+00:00,-24.56
47287,2020-05-24 06:00:00+00:00,-23.92


In [7]:
#Unusually low electricity demand
print("Load < 10,000 MW:",(df[LOAD] < 10000).sum())
print("Load < 15,000 MW:", (df[LOAD] < 15000).sum())
print("Mean load:",round(df[LOAD].mean()))
df.nsmallest(10,LOAD)[["utc_timestamp",LOAD]]



Load < 10,000 MW: 27
Load < 15,000 MW: 84
Mean load: 34921


,utc_timestamp,GB_GBN_load_actual_entsoe_transparency
38424,2019-05-20 23:00:00+00:00,511.0
38425,2019-05-21 00:00:00+00:00,539.0
38426,2019-05-21 01:00:00+00:00,575.0
39981,2019-07-24 20:00:00+00:00,757.0
39982,2019-07-24 21:00:00+00:00,852.0
38997,2019-06-13 20:00:00+00:00,874.0
39980,2019-07-24 19:00:00+00:00,926.0
38996,2019-06-13 19:00:00+00:00,1029.0
40005,2019-07-25 20:00:00+00:00,1282.0
40004,2019-07-25 19:00:00+00:00,1476.0


In [8]:
#Flag potential outliers
def flag_outliers(data):
    out = data.copy()
    q1, q3 = out[P].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    out["flag_price_negative"] = out[P] < 0
    out["flag_price_high"] = out[P] >= 500
    out["flag_price_iqr"] = (out[P] < lo) | (out[P] > hi)
    out["flag_solar_profile_gt1"] = out["GB_GBN_solar_profile"] > 1
    out["flag_solar_gt_capacity"] = out[SOLAR] > out[SOLAR_CAP]
    out["flag_load_low"] = out[LOAD] < 10000
    return out


def outlier_summary(data):
    cols = [c for c in data.columns if c.startswith("flag_")]
    s = data[cols].sum().rename("rows_flagged").to_frame()
    s["percent"] = (s["rows_flagged"] / len(data) * 100).round(2)
    return s

flagged = flag_outliers(df)
print("Shape before:", df.shape, "| after:", flagged.shape)
outlier_summary(flagged)

Shape before: (50401, 16) | after: (50401, 22)


,rows_flagged,percent
flag_price_negative,81,0.16
flag_price_high,10,0.02
flag_price_iqr,1804,3.58
flag_solar_profile_gt1,233,0.46
flag_solar_gt_capacity,233,0.46
flag_load_low,27,0.05


## Cleaning decisions and justification 

| Issue | Rows | Decision | Justification |
|---|---|---|---|
| Duplicate rows / timestamps | 0 | No action | None found |
| Negative demand, generation, capacity | 0 | No action | None found |
| Negative day-ahead prices | 81 | Keep and flag | Negative prices occur when supply is higher than demand |
| Prices >= 500 | 10 | Keep and flag | Looks like a genuine market spike; to be verified |
| Prices outside IQR bounds | 1,804 | Keep and flag | Statistical outliers only; many are probably real market conditions |
| Solar profile > 1 / solar > capacity | 233 | Keep and flag | The profile is a calculated ratio; the capacity value may be outdated. Not capped |
| Load < 10,000 MW | 27 | Keep and flag | Possibly faulty readings; analysts should exclude flagged rows from share calculations |